# Architecture Inspection — HubertMultiTask

Walks through every stage of the new pronunciation-scoring pipeline with concrete tensor shapes.
All cells use **synthetic data** — no dataset download needed.

**Pipeline**
```
Raw audio
  └─ HuBERT encoder  (output_hidden_states=True)
       └─ Layer-weighted sum  (learnable softmax weights × 13 layers)   → (B, T, H=768)
            └─ Linear projection                                          → (B, T, d_model=256)
                                                      ↘  Query (Q)
Reference text                               Cross-attention fusion       → (B, T, d_model)
  └─ BERT encoder                                     ↗  Key / Value (K/V)
       └─ CLS token                      (optional: +phone embeds from CTCPhoneAligner)
            └─ Linear+LayerNorm          → (B, 1, d_model)

  └─ Post-fusion Transformer block (pre-LN, 2 layers)                    → (B, T, d_model)
  └─ Mean pool over time                                                  → (B, d_model)
  └─ MLP scoring head: FC → ReLU → Dropout → FC → Sigmoid × 5           → (B, 5) ∈ [0,1]
       └─ Auxiliary prosody head: FC → ReLU → FC                         → (B, 5)  (unbounded)
```

## 0 · Imports & path setup

In [1]:
import sys, os

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import torch
import torch.nn.functional as F
import numpy as np

print('torch:', torch.__version__)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

torch: 2.10.0+cu126
device: cuda


## 1 · Instantiate the model

In [2]:
from models.hubert_multitask import HubertMultiTask
from models.constants import SENT_DIMS, PROSODY_DIMS

model = HubertMultiTask(
    model_name='facebook/hubert-base-ls960',
    d_model=256,
    num_heads=8,
    num_transformer_layers=2,
    dropout=0.1,
    freeze_fe=True,                          # HuBERT CNN feature extractor frozen
    text_model_name='Qwen/Qwen3-Embedding-0.6B',
    freeze_text_encoder=True,                # Qwen3 weights frozen
    prosody_feat_dim=len(PROSODY_DIMS),
).to(DEVICE)
model.eval()
print(model)

d:\pronunciation_scoring\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 310/310 [00:00<00:00, 885.33it/s, Materializing param=norm.weight]                              
d:\pronunciation_scoring\models\hubert_multitask.py:127: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


HubertMultiTask(
  (hubert): HubertModel(
    (feature_extractor): HubertFeatureEncoder(
      (conv_layers): ModuleList(
        (0): HubertGroupNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        )
        (1-4): 4 x HubertNoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x HubertNoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): HubertFeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): HubertEncod

### 1.1 · Parameter count by submodule

In [13]:
def param_count(module):
    total     = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return total, trainable

rows = [
    ('hubert (CNN frozen, transformer trainable)', model.hubert),
    ('  layer_weights  (13 scalars)',              None),   # handled separately
    ('audio_proj',                                 model.audio_proj),
    ('text_encoder / Qwen',               model.text_encoder),
    ('text_proj  (Linear + LayerNorm)',            model.text_proj),
    ('cross_attn_fusion  (MHA + LN)',              model.cross_attn_fusion),
    ('transformer  (2× pre-LN encoder layers)',    model.transformer),
    ('scorer  (MLP head)',                         model.scorer),
    ('prosody_feat_head  (auxiliary)',             model.prosody_feat_head),
]

print(f"{'Module':<48} {'Total params':>14} {'Trainable':>12}")
print('─' * 76)
for name, mod in rows:
    if mod is None:
        t  = model.layer_weights.numel()
        tr = t if model.layer_weights.requires_grad else 0
    else:
        t, tr = param_count(mod)
    print(f"  {name:<46} {t:>14,} {tr:>12,}")

t_all, tr_all = param_count(model)
print('─' * 76)
print(f"  {'TOTAL':<46} {t_all:>14,} {tr_all:>12,}")

Module                                             Total params    Trainable
────────────────────────────────────────────────────────────────────────────
  hubert (CNN frozen, transformer trainable)         94,371,712   90,171,264
    layer_weights  (13 scalars)                              13           13
  audio_proj                                            196,864      196,864
  text_encoder / Qwen                               595,776,512            0
  text_proj  (Linear + LayerNorm)                       262,912      262,912
  cross_attn_fusion  (MHA + LN)                         263,680      263,680
  transformer  (2× pre-LN encoder layers)             1,579,520    1,579,520
  scorer  (MLP head)                                     67,077       67,077
  prosody_feat_head  (auxiliary)                         67,077       67,077
────────────────────────────────────────────────────────────────────────────
  TOTAL                                             692,585,367   92,608,407

## 2 · Synthetic batch

Mimics a real `Collator` output — no dataset download needed.

In [4]:
from transformers import AutoTokenizer

torch.manual_seed(0)

B         = 2
T_SAMPLES = 32000   # 2-second clip @ 16 kHz

# Waveform (output of Wav2Vec2FeatureExtractor)
input_values   = torch.randn(B, T_SAMPLES, device=DEVICE)
attention_mask = torch.ones(B, T_SAMPLES, dtype=torch.long, device=DEVICE)

# Reference transcripts (what the learner was supposed to say)
TEXTS = [
    "the quick brown fox jumps over the lazy dog",
    "she sells seashells by the seashore",
]

# Tokenise for BERT text stream
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
text_enc  = tokenizer(
    TEXTS, padding=True, truncation=True, max_length=32, return_tensors='pt'
)
text_input_ids      = text_enc['input_ids'].to(DEVICE)       # (B, L)
text_attention_mask = text_enc['attention_mask'].to(DEVICE)  # (B, L)

# Sentence targets and prosody features (collator outputs, used for loss only)
prosody_feats = torch.rand(B, 5, device=DEVICE)
sent_targets  = torch.rand(B, 5, device=DEVICE)

print('input_values        :', input_values.shape)
print('text_input_ids      :', text_input_ids.shape)
print('  tokens sample 0   :', text_input_ids[0].tolist())
print('text_attention_mask :', text_attention_mask.shape)
print('prosody_feats       :', prosody_feats.shape)

input_values        : torch.Size([2, 32000])
text_input_ids      : torch.Size([2, 11])
  tokens sample 0   : [101, 1996, 4248, 2829, 4419, 14523, 2058, 1996, 13971, 3899, 102]
text_attention_mask : torch.Size([2, 11])
prosody_feats       : torch.Size([2, 5])


## 3 · Layer-weighted HuBERT encoding

HuBERT produces one hidden-state tensor per transformer layer (+ the initial embedding layer output).
The model learns a **softmax-normalised scalar weight** for each of the 13 layers, then sums them.

- Early layers tend to capture **acoustic / phonetic** detail.
- Later layers capture **more abstract linguistic** structure.
- The weights are initialised uniform (1/13) and sharpened by training.

> Compare to `output = model.hubert(x).last_hidden_state` — that uses only layer 12 and discards all earlier representations.

In [5]:
with torch.no_grad():
    raw_out    = model.hubert(input_values, output_hidden_states=True)
    all_layers = torch.stack(raw_out.hidden_states, dim=0)    # (n_layers, B, T, H)
    layer_w    = torch.softmax(model.layer_weights, dim=0)    # (n_layers,)  sums to 1
    hidden     = (layer_w[:, None, None, None] * all_layers).sum(dim=0)  # (B, T, H)

n_layers, B_, T_frames, H = all_layers.shape
print(f'HuBERT hidden states : {list(all_layers.shape)}  ({n_layers} layers × B × T × H)')
print(f'Weighted-sum output  : {list(hidden.shape)}')
print(f'Frame rate           : ~{T_frames / (T_SAMPLES / 16000):.1f} Hz  (expected ~50 Hz)')
print()
print('Layer weights (softmax-normalised, init ≈ uniform 1/13 = 0.0769):')
for i, w in enumerate(layer_w.tolist()):
    bar = '█' * max(1, int(w * 500))
    print(f'  layer {i:>2d} : {w:.4f}  {bar}')

HuBERT hidden states : [13, 2, 99, 768]  (13 layers × B × T × H)
Weighted-sum output  : [2, 99, 768]
Frame rate           : ~49.5 Hz  (expected ~50 Hz)

Layer weights (softmax-normalised, init ≈ uniform 1/13 = 0.0769):
  layer  0 : 0.0769  ██████████████████████████████████████
  layer  1 : 0.0769  ██████████████████████████████████████
  layer  2 : 0.0769  ██████████████████████████████████████
  layer  3 : 0.0769  ██████████████████████████████████████
  layer  4 : 0.0769  ██████████████████████████████████████
  layer  5 : 0.0769  ██████████████████████████████████████
  layer  6 : 0.0769  ██████████████████████████████████████
  layer  7 : 0.0769  ██████████████████████████████████████
  layer  8 : 0.0769  ██████████████████████████████████████
  layer  9 : 0.0769  ██████████████████████████████████████
  layer 10 : 0.0769  ██████████████████████████████████████
  layer 11 : 0.0769  ██████████████████████████████████████
  layer 12 : 0.0769  ██████████████████████████████████████


## 4 · Audio linear projection  →  d_model

`Linear(H=768, d_model=256)` maps the weighted HuBERT output to the shared projection dimension.

This is the **Query (Q)** sequence fed into the cross-attention fusion block.

In [6]:
with torch.no_grad():
    audio_emb = model.audio_proj(hidden)   # (B, T, d_model)

d_model = audio_emb.shape[-1]
print(f'hidden    (layer-weighted HuBERT) : {list(hidden.shape)}')
print(f'audio_emb (after Linear proj)     : {list(audio_emb.shape)}')
print(f'  H → d_model  :  {H} → {d_model}  (compression ratio {H/d_model:.1f}×)')
print(f'  value range  : [{audio_emb.min():.3f}, {audio_emb.max():.3f}]')

hidden    (layer-weighted HuBERT) : [2, 99, 768]
audio_emb (after Linear proj)     : [2, 99, 256]
  H → d_model  :  768 → 256  (compression ratio 3.0×)
  value range  : [-0.385, 0.383]


## 5 · Text path — Qwen3-Embedding → mean pool → Linear+LN  →  d_model

**Qwen3-Embedding-0.6B** is a decoder-style embedding model (~0.6 B parameters, `hidden_size = 1024`).
Unlike BERT, it has no special CLS token, so we use **mask-aware mean pooling** over all non-padding
token embeddings — matching the recommended usage in the snippet above.

```python
# Mask-aware mean pool (handles variable-length padding correctly)
mask       = attention_mask.unsqueeze(-1).float()   # (B, L, 1)
embeddings = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
```

The resulting `(B, text_H=1024)` vector is projected to `(B, d_model=256)` and unsqueezed
to `(B, 1, 256)` — the **Key / Value** for cross-attention.

> `text_proj` (`Linear(1024 → 256) + LayerNorm`) bridges the Qwen3 hidden size to the
> shared `d_model` used by both streams.

In [7]:
with torch.no_grad():
    text_out = model.text_encoder(text_input_ids, text_attention_mask)
    hidden_t = text_out.last_hidden_state                              # (B, L, text_H)

    # Mask-aware mean pooling
    mask     = text_attention_mask.unsqueeze(-1).float()               # (B, L, 1)
    text_emb = (hidden_t * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)  # (B, text_H)

    text_kv  = model.text_proj(text_emb).unsqueeze(1)                 # (B, 1, d_model)

text_H = text_emb.shape[-1]
print(f'Qwen3 last_hidden_state : {list(hidden_t.shape)}')
print(f'After mask-aware mean pool : {list(text_emb.shape)}  (text_H = {text_H})')
print(f'After text_proj        : {list(model.text_proj(text_emb).shape)}  (text_H → d_model)')
print(f'Key/Value tensor       : {list(text_kv.shape)}   ← single K/V token per sample')
print()
cos_sim = torch.nn.functional.cosine_similarity(text_emb[0], text_emb[1], dim=0)
print(f'Cosine similarity between the two sentence embeddings: {cos_sim:.4f}')

Qwen3 last_hidden_state : [2, 11, 1024]
After mask-aware mean pool : [2, 1024]  (text_H = 1024)
After text_proj        : [2, 256]  (text_H → d_model)
Key/Value tensor       : [2, 1, 256]   ← single K/V token per sample

Cosine similarity between the two sentence embeddings: 0.7561


## 6 · Cross-attention fusion  (Audio Q, Text K/V)

`CrossAttentionFusion` performs multi-head attention where:
- **Query** = audio frame embeddings `(B, T, d_model)` — *what does audio want to know?*
- **Key / Value** = text CLS token `(B, 1, d_model)` — *what was said?*

Every audio frame independently queries the text representation. With a single K/V token all
frames receive the same text context injection, but scaled by their own query projection.
A **residual connection + LayerNorm** follows.

The shift vector `fused − audio_emb` shows how much text moved each frame.

In [8]:
with torch.no_grad():
    fused = model.cross_attn_fusion(audio_emb, text_kv)   # (B, T, d_model)

print(f'audio_emb (Q, pre-fusion)  : {list(audio_emb.shape)}')
print(f'text_kv   (K/V, 1 token)   : {list(text_kv.shape)}')
print(f'fused     (post-fusion)    : {list(fused.shape)}')
print()

# L2 shift: how far each frame moved in embedding space after text injection
shift = (fused - audio_emb).norm(dim=-1)    # (B, T)
cos   = torch.nn.functional.cosine_similarity(fused, audio_emb, dim=-1)  # (B, T)

print(f"{'':>10}  {'L2 shift':>12}  {'cosine sim':>12}")
print(f"{'':>10}  {'mean / max':>12}  {'mean / min':>12}")
print('─' * 42)
for b in range(B):
    print(f"  batch {b} :  "
          f"{shift[b].mean():.4f} / {shift[b].max():.4f}  "
          f"{cos[b].mean():.4f} / {cos[b].min():.4f}")

audio_emb (Q, pre-fusion)  : [2, 99, 256]
text_kv   (K/V, 1 token)   : [2, 1, 256]
fused     (post-fusion)    : [2, 99, 256]

                L2 shift    cosine sim
              mean / max    mean / min
──────────────────────────────────────────
  batch 0 :  15.6702 / 15.8574  0.2489 / 0.1372
  batch 1 :  15.6532 / 15.7800  0.2618 / 0.1885


## 7 · Post-fusion Transformer block  (Conformer-style refinement)

A **pre-LN `TransformerEncoder`** (2 layers, `dim_feedforward = 4 × d_model = 1024`) runs
self-attention over the full fused audio sequence.

- Pre-norm: each sub-layer normalises *before* projection → more stable gradients.
- Self-attention allows every frame to attend to all others **after** text context has been injected,
  enabling the model to reason about pronunciation coherence across the utterance.
- Inspection: the attention weight matrix `(B, T, T)` shows which frames influence each other.

In [9]:
with torch.no_grad():
    refined = model.transformer(fused)   # (B, T, d_model)

print(f'fused   (input)  : {list(fused.shape)}')
print(f'refined (output) : {list(refined.shape)}')
print()

shift = (refined - fused).norm(dim=-1)
print('Per-frame L2 shift through Transformer (fused → refined):')
for b in range(B):
    print(f'  batch {b} — mean {shift[b].mean():.4f}  '
          f'min {shift[b].min():.4f}  max {shift[b].max():.4f}')

# Self-attention pattern in the first transformer layer
tl = model.transformer.layers[0]
with torch.no_grad():
    x_norm = tl.norm1(fused)
    _, attn_w = tl.self_attn(x_norm, x_norm, x_norm,
                              need_weights=True, average_attn_weights=True)

print(f'\nTransformer layer-0 self-attention map : {list(attn_w.shape)}  (B, T, T)')
print(f'  batch 0, frame 0 → entropy : '
      f'{-(attn_w[0, 0] * (attn_w[0, 0] + 1e-9).log()).sum():.4f} nats '
      f'(max entropy = {torch.tensor(float(T_frames)).log():.4f})')

fused   (input)  : [2, 99, 256]
refined (output) : [2, 99, 256]

Per-frame L2 shift through Transformer (fused → refined):
  batch 0 — mean 14.8660  min 14.7245  max 15.0343
  batch 1 — mean 14.8225  min 14.6780  max 14.9816

Transformer layer-0 self-attention map : [2, 99, 99]  (B, T, T)
  batch 0, frame 0 → entropy : 4.5946 nats (max entropy = 4.5951)


## 8 · Utterance-level mean pooling  +  MLP scoring head

The refined sequence is collapsed to a single utterance vector via **mean pooling over time**,
then passed through the two-layer MLP head:

```
FC(d_model → d_model) → ReLU → Dropout → FC(d_model → 5) → Sigmoid
```

Sigmoid maps each of the 5 output dimensions to `[0, 1]`, matching the normalised SpeechOcean
labels (`score / 10`).  Multiply by 10 to recover the original 0–10 scale.

The **auxiliary prosody head** predicts audio-level prosody features from the same `utt_emb`
and provides a training regularisation signal (weight `w_pfeat = 0.1` in `compute_loss`).

In [10]:
with torch.no_grad():
    utt_emb      = refined.mean(dim=1)           # (B, d_model)
    sent_pred    = model.scorer(utt_emb)          # (B, 5)
    prosody_pred = model.prosody_feat_head(utt_emb)  # (B, 5)

print(f'refined      : {list(refined.shape)}    (B, T, d_model)')
print(f'utt_emb      : {list(utt_emb.shape)}    ← mean over T={refined.shape[1]} frames')
print(f'sent_pred    : {list(sent_pred.shape)}    ← Sigmoid → all in [0, 1]  (×10 for display)')
print(f'prosody_pred : {list(prosody_pred.shape)}    ← auxiliary, unbounded')
print()
print(f"{'aspect':<15}  {'pred (×10)':>10}  {'target (×10)':>13}")
print('─' * 44)
for i, dim in enumerate(SENT_DIMS):
    print(f"  {dim:<13}  {sent_pred[0,i]*10:>10.3f}  {sent_targets[0,i]*10:>13.3f}")
print()
print(f"{'prosody dim':<15}  {'pred':>8}")
print('─' * 28)
for i, dim in enumerate(PROSODY_DIMS):
    print(f"  {dim:<13}  {prosody_pred[0,i]:>8.4f}")

refined      : [2, 99, 256]    (B, T, d_model)
utt_emb      : [2, 256]    ← mean over T=99 frames
sent_pred    : [2, 5]    ← Sigmoid → all in [0, 1]  (×10 for display)
prosody_pred : [2, 5]    ← auxiliary, unbounded

aspect           pred (×10)   target (×10)
────────────────────────────────────────────
  total               3.715          0.194
  accuracy            5.317          7.733
  fluency             4.171          8.650
  prosodic            4.461          8.097
  completeness        7.053          6.666

prosody dim          pred
────────────────────────────
  rms              0.2753
  zcr              0.1787
  peak_rate       -0.5982
  f0_mean          0.1036
  f0_std           0.1360


## 9 · Full model forward pass — all heads together

In [11]:
with torch.no_grad():
    outputs = model(
        input_values=input_values,
        attention_mask=attention_mask,
        text_input_ids=text_input_ids,
        text_attention_mask=text_attention_mask,
        # Legacy kwargs (prosody_feats, word_mask, etc.) are absorbed by **_
        prosody_feats=prosody_feats,
    )

print('Output tensors:')
for key, val in outputs.items():
    print(f'  {key:<15} : {str(list(val.shape)):<20}  range [{val.min():.4f}, {val.max():.4f}]')

Output tensors:
  sent_pred       : [2, 5]                range [0.3715, 0.7053]
  prosody_pred    : [2, 5]                range [-0.5982, 0.2753]


## 10 · Loss computation

`compute_loss` applies **Smooth L1** on the sentence head and plain **L1** on the auxiliary
prosody head.  Word / phone head weights are set to 0 since those heads were removed.

In [12]:
from models.train import compute_loss

batch = {
    'sent_targets':  sent_targets,
    'prosody_feats': prosody_feats,
    # word/phone fields are still expected in the batch dict signature; zero them out
    'word_scores':   torch.zeros(B, 8,     device=DEVICE),
    'word_mask':     torch.zeros(B, 8,     dtype=torch.bool, device=DEVICE),
    'phone_scores':  torch.zeros(B, 8, 10, device=DEVICE),
    'phone_mask':    torch.zeros(B, 8, 10, dtype=torch.bool, device=DEVICE),
}

total_loss, logs = compute_loss(
    outputs, batch,
    w_sent=1.0, w_pfeat=0.1,
    w_words=0.0, w_phone=0.0,    # word/phone heads removed
)

print(f'Total loss : {total_loss.item():.6f}')
print()
for k, v in logs.items():
    print(f'  {k:<28} {v:.6f}')

TypeError: compute_loss() got an unexpected keyword argument 'w_words'

## 11 · Training step — gradient coverage check

Verifies that gradients flow back through every **trainable** component of the new pipeline.

Frozen modules (HuBERT CNN feature extractor, BERT encoder) should show `no_grad > 0`,
and their `has_grad` count should be 0.

In [ ]:
import torch.nn.functional as F

model.train()
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=2e-5
)
optimizer.zero_grad()

out = model(
    input_values=input_values,
    attention_mask=attention_mask,
    text_input_ids=text_input_ids,
    text_attention_mask=text_attention_mask,
)
loss  = F.smooth_l1_loss(out['sent_pred'],    sent_targets)
loss += 0.1 * F.l1_loss(out['prosody_pred'], prosody_feats)
loss.backward()

grad_report = {}
no_grad     = {}
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    top      = '.'.join(name.split('.')[:2])
    has_grad = param.grad is not None and param.grad.abs().max().item() > 1e-12
    grad_report[top] = grad_report.get(top, 0) + int(has_grad)
    no_grad[top]     = no_grad.get(top, 0)     + int(not has_grad)

print(f'Loss : {loss.item():.6f}\n')
print(f'{"":2} {"Submodule":<40}  {"has_grad":>9}  {"no_grad":>9}')
print('─' * 66)
for key in sorted(set(grad_report) | set(no_grad)):
    g  = grad_report.get(key, 0)
    ng = no_grad.get(key, 0)
    ok = '✓' if ng == 0 else '⚠'
    print(f'  {ok} {key:<40}  {g:>9}  {ng:>9}')

optimizer.step()
model.eval()
print('\nStep complete.')

## 12 · Module repr — full scoring stack

In [ ]:
for label, mod in [
    ('audio_proj         (H → d_model)',       model.audio_proj),
    ('text_proj          (text_H → d_model)',  model.text_proj),
    ('cross_attn_fusion  (Audio Q, Text K/V)', model.cross_attn_fusion),
    ('transformer        (post-fusion block)',  model.transformer),
    ('scorer             (MLP head)',           model.scorer),
    ('prosody_feat_head  (auxiliary)',          model.prosody_feat_head),
]:
    print(f'=== {label} ===')
    print(mod)
    print()